# 💬 Build Your Own AI Chatbot with Gemini Flash

**Goal:** Build a chatbot that can answer **any question** you ask, with memory, a custom personality, and a real chat window, using Google's **Gemini 3.5 Flash** model.

### What you will build
1. A first AI response in **3 lines of code**
2. A chatbot that **streams** answers word by word
3. A chatbot with **memory** (remembers what you said earlier)
4. A chatbot with a **personality** (system instructions)
5. A real **chat interface** (like ChatGPT) using Gradio

### What you need
- A Google account
- A **free Gemini API key** (we'll get it in Step 1, takes 1 minute)

> This continues from Day 3: the model behind the chatbot is a **Transformer**, the same family as DistilBERT. DistilBERT *understands* text, while Gemini *generates* text.

---
## 💡 How does a chatbot work?

```
  You type a question
        │
        ▼
 ┌─────────────────┐     internet      ┌──────────────────────┐
 │  Your Colab code │ ───────────────▶ │ Gemini model (Google) │
 │  (the "client")  │ ◀─────────────── │ runs on Google's GPUs │
 └─────────────────┘   generated text   └──────────────────────┘
        │
        ▼
  Answer shown to you
```

Key ideas:
- **The model is not on your computer.** It is too big. Your code sends a request to Google's servers through an **API** (a way for programs to talk to each other) and gets text back.
- **An API key** is your password that tells Google who is making the request.
- **LLMs have no memory by themselves.** Every request is independent. A chatbot "remembers" because we **send the whole conversation again** with each new message.
- The model works by **predicting the next token**, again and again, which is how it writes full answers.

---
## 🔑 Step 1: Get your free API key

1. Open **https://aistudio.google.com/apikey**
2. Sign in with your Google account
3. Click **Create API key** and copy it

### Store it safely in Colab Secrets 🔒
1. In Colab, click the **🔑 key icon** in the left sidebar ("Secrets")
2. Click **Add new secret**
3. Name: `GEMINI_API_KEY`   Value: *(paste your key)*
4. Turn on **Notebook access**

> ⚠️ **Never paste your API key directly into code** or share a notebook that contains it. Anyone with your key can use your quota. Secrets keep it hidden.

---
## 📦 Step 2: Install the library and connect

We use Google's official **`google-genai`** library.

In [1]:
!pip install -q -U google-genai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 882.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


In [11]:
from google import genai
from google.genai import types

import getpass

def load_api_key():
    # Get the Gemini API key from Colab Secrets, or ask you to paste it.
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass                                   # no secret found, we will ask below

    # Some setups return a dictionary instead of plain text. Pull the key out of it.
    if isinstance(key, dict):
        text_values = [v for v in key.values() if isinstance(v, str)]
        key = next((v for v in text_values if v.startswith("AIza")),
                   text_values[0] if text_values else None)

    # Still nothing usable? Ask for it (the text stays hidden as you type).
    if not isinstance(key, str) or not key.strip():
        key = getpass.getpass("Paste your Gemini API key: ")

    return key.strip()

API_KEY = load_api_key()
client = genai.Client(api_key=API_KEY)

# 👇 The model we will use. To try a newer one, just change this one line.
MODEL_NAME = "gemini-3.1-flash-lite"

print("Connected! Using model:", MODEL_NAME)

Connected! Using model: gemini-3.1-flash-lite


### 🔍 (Optional) Which models can my key use?
Google releases new models often. If you ever see a *"model not found"* error, run this cell to see the exact names available to you.

In [12]:
for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


---
## 🚀 Step 3: Your first AI response

One function call: `generate_content()`. We give it the **model** and our **prompt** (the question).

In [13]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain what a neural network is in 3 simple sentences."
)

print(response.text)

A neural network is a type of computer program designed to mimic the way a human brain processes information. It uses layers of interconnected nodes to identify patterns and relationships within large sets of data. By learning from examples, the network improves its ability to make accurate predictions or decisions over time.


### 🧪 Try it yourself
Change the `contents` text. Ask anything: a science question, a joke, a recipe, help with homework.

### 🔢 Tokens: what the model actually counts
Remember tokens from Day 3? The API tells us how many tokens were used. This is also how AI companies **charge** for usage.

In [14]:
usage = response.usage_metadata
print("Tokens in your prompt:   ", usage.prompt_token_count)
print("Tokens in the answer:    ", usage.candidates_token_count)
print("Total tokens:            ", usage.total_token_count)

Tokens in your prompt:    13
Tokens in the answer:     56
Total tokens:             69


---
## ⚡ Step 4: Streaming, answers appear word by word

Waiting for a long answer to finish feels slow. ChatGPT shows text as it is written. We can do the same with `generate_content_stream()`, which gives us the answer in small **chunks**.

In [15]:
stream = client.models.generate_content_stream(
    model=MODEL_NAME,
    contents="Write a short, fun story about a robot learning to cook."
)

for chunk in stream:
    print(chunk.text or "", end="", flush=True)

Unit 742, known to his human family as "Chip," was designed for high-precision welding. He was excellent at melting steel, but he was currently facing his greatest challenge: a lemon meringue pie.

"It’s all about the *feel*, Chip," Sarah, his owner, said, gesturing at the bowl of egg whites. "You have to whip them until they form stiff peaks. It’s like magic."

Chip processed this. *Magic: A non-scientific phenomenon involving irrational outcomes.* 

He extended his titanium whisk attachment. With a mechanical hum, he began to rotate. The recipe called for "gentle folding," but Chip’s default setting was "industrial oscillation." Within three seconds, the egg whites were not only stiff; they were vibrating at a frequency that caused the mixing bowl to hover an inch off the counter.

"Whoa! Slow down, big guy!" Sarah laughed, hitting the override button.

Chip retracted his whisk, his optic sensors dimming in what he hoped registered as embarrassment. "Apologies, Sarah. My internal gyr

---
## 🧠 Step 5: Give the chatbot memory

Let's prove that a plain request has **no memory**:

In [16]:
r1 = client.models.generate_content(model=MODEL_NAME, contents="My name is Arjun and I love cricket.")
print("Bot:", r1.text)

r2 = client.models.generate_content(model=MODEL_NAME, contents="What is my name?")
print("\nBot:", r2.text)

Bot: Hello Arjun! It’s great to meet a fellow cricket fan. 

Since you love cricket, I’m curious—what’s your favorite part of the game? Are you a fan of the tactical test of Test cricket, the high-intensity drama of T20s, or the classic rhythm of ODIs?

Also, who is your all-time favorite player? Are you a fan of a specific team in the IPL or international circuit?

Bot: I don’t know your name. As an AI, I don’t have access to your personal identity, private information, or your camera/microphone unless you explicitly provide that information during our conversation. 

If you'd like me to call you by a certain name, feel free to tell me!


😮 The bot doesn't know! Each call is a brand-new conversation.

### The fix: a chat session
`client.chats.create()` makes a **chat object** that automatically stores the conversation and sends the full history with every new message.

In [17]:
chat = client.chats.create(model=MODEL_NAME)

r1 = chat.send_message("My name is Arjun and I love cricket.")
print("Bot:", r1.text)

r2 = chat.send_message("What is my name, and what sport do I like?")
print("\nBot:", r2.text)

Bot: Hi Arjun! It’s great to meet you. Since you love cricket, you’re in good company!

Who is your favorite cricketer, or do you have a favorite team that you follow? Whether it's the thrill of a Test match, the intensity of an ODI, or the excitement of T20s, there’s always something interesting to talk about in the world of cricket. 

Are you playing at the moment, or are you just a passionate fan?

Bot: Your name is **Arjun**, and you love **cricket**!


✅ Now it remembers! Let's look at what is stored behind the scenes. **This list is the "memory".**

In [18]:
for message in chat.get_history():
    text = "".join(part.text or "" for part in message.parts)
    print(f"[{message.role.upper()}]  {text[:150]}")
    print()

[USER]  My name is Arjun and I love cricket.

[MODEL]  Hi Arjun! It’s great to meet you. Since you love cricket, you’re in good company!

Who is your favorite cricketer, or do you have a favorite team that

[USER]  What is my name, and what sport do I like?

[MODEL]  Your name is **Arjun**, and you love **cricket**!



### 🧐 Important insight
The model itself learned nothing. We simply **resend the whole history** every time. This means:
- Long conversations use **more tokens** (and cost more)
- Models have a **context window**, a maximum amount of text they can handle at once
- If you start a new chat object, the memory is gone

---
## 🎭 Step 6: Give your chatbot a personality

A **system instruction** tells the model *how to behave* for the whole conversation: its role, tone and rules. Users don't see it, but the model follows it.

In [19]:
teacher_config = types.GenerateContentConfig(
    system_instruction=(
        "You are a friendly and patient AI teacher for school and college students. "
        "Explain things in simple language with a real-life example. "
        "Keep answers short (maximum 5 sentences) unless asked for more detail. "
        "End every answer with one question that makes the student think."
    )
)

teacher_chat = client.chats.create(model=MODEL_NAME, config=teacher_config)

response = teacher_chat.send_message("What is machine learning?")
print(response.text)

Machine learning is a way of teaching computers to learn from patterns and data rather than following strict, pre-written rules. Instead of telling a computer exactly what to do, you feed it examples so it can figure out the "rules" on its own. A real-life example is how Netflix suggests movies you might like based on the shows you have already watched. The more you watch, the better the system gets at understanding your personal taste. If a computer can learn from experience, how do you think it might change the way we study in the future?


### 🧪 Try it yourself, change the personality!
Replace the `system_instruction` with one of these and ask the same question:
- *"You are a pirate. Answer everything like a pirate."* 🏴‍☠️
- *"You are a Shakespeare-style poet. Reply only in rhyming verse."*
- *"You are a strict interviewer. Respond to every answer with a tough follow-up."*
- *"Explain everything as if the user is 5 years old."*

The same model gives completely different behaviour from one paragraph of instructions. This is called **prompt engineering**.

---
## 💻 Step 7: A chatbot loop

Now let's make a real conversation. This loop keeps asking for your input, sends it to the model, and prints the streamed answer. Type **`quit`** to stop.

Run the cell, then type in the box that appears below it.

In [20]:
# A general-purpose assistant: it will answer whatever you ask
assistant_config = types.GenerateContentConfig(
    system_instruction=(
        "You are a helpful, honest and friendly AI assistant. "
        "Answer any question clearly and accurately. "
        "If you are not sure about something, say so instead of guessing."
    )
)

chat = client.chats.create(model=MODEL_NAME, config=assistant_config)

print("🤖 Chatbot ready! Type your question (or 'quit' to stop).\n")

while True:
    user_input = input("You: ")

    if user_input.strip().lower() in ("quit", "exit", "bye"):
        print("Bot: Goodbye! 👋")
        break
    if not user_input.strip():
        continue

    print("Bot: ", end="")
    try:
        for chunk in chat.send_message_stream(user_input):
            print(chunk.text or "", end="", flush=True)
    except Exception as e:
        print(f"\n⚠️ Something went wrong: {e}")
    print("\n")

🤖 Chatbot ready! Type your question (or 'quit' to stop).

You: hi
Bot: Hello! How can I help you today?

You: how is the weather tommorow
Bot: To give you an accurate weather forecast, I need to know your **city or location**. 

Could you please tell me where you are located?



KeyboardInterrupt: Interrupted by user

---
## 🖥️ Step 8: A real chat window with Gradio

A text loop is good for learning, but a real chatbot needs an interface. **Gradio** builds a ChatGPT-style web page in a few lines.

How it works: Gradio calls our function `respond(message, history)` every time you send a message. We turn the history into the format Gemini expects and **stream** the answer back.

In [21]:
import gradio as gr

def history_to_contents(history, message):
    """Convert Gradio's chat history + the new message into Gemini's format."""
    contents = []

    def add(role, text):
        if text:
            contents.append(types.Content(role=role, parts=[types.Part(text=text)]))

    for turn in history:
        if isinstance(turn, dict):                       # newer Gradio: {"role":..., "content":...}
            role = "user" if turn.get("role") == "user" else "model"
            text = turn.get("content", "")
            if isinstance(text, list):                   # content can be a list of parts
                text = " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in text)
            add(role, text)
        else:                                            # older Gradio: (user, bot) pairs
            add("user", turn[0])
            add("model", turn[1])

    add("user", message)
    return contents


def respond(message, history):
    try:
        stream = client.models.generate_content_stream(
            model=MODEL_NAME,
            contents=history_to_contents(history, message),
            config=assistant_config,
        )
        answer = ""
        for chunk in stream:
            answer += chunk.text or ""
            yield answer                                  # show the text as it grows
    except Exception as e:
        yield f"⚠️ Sorry, something went wrong: {e}"


demo = gr.ChatInterface(
    fn=respond,
    title="🤖 My Gemini Chatbot",
    description="Ask me anything!",
    examples=["Explain black holes simply", "Give me a 3-day study plan for exams", "Write a Python function to reverse a string"],
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://757d5f91e2b0ff96a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://757d5f91e2b0ff96a9.gradio.live


🎉 You now have a working chatbot with a chat window!

> ⚠️ **Heads-up:** while this cell is running, Gradio may print a **public link**. Anyone who opens that link uses **your API key's quota**. Share it only for the demo, then **stop the cell** (the ⏹ button) when finished.

### 🧪 Customize it
- Change `title` and `description`
- Change `assistant_config` to make a **specialist bot**: a cooking assistant, a fitness coach, a career counsellor, a bot for your college
- Add your own `examples`

---
## 🌐 Bonus: Let the chatbot search the web

LLMs only know what they learned during training, so they can be **out of date**. **Grounding with Google Search** lets the model look things up and answer using current information.

In [23]:
search_tool = types.Tool(google_search=types.GoogleSearch())

grounded_config = types.GenerateContentConfig(tools=[search_tool])

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="What are the latest developments in AI this week?",
    config=grounded_config,
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

Compare this to the same question **without** the search tool. Which answer is more current?

---
## ⚠️ Limitations: what every chatbot builder must know

| Issue | What it means |
|---|---|
| **Hallucination** | The model can state wrong things confidently. It predicts likely text, not verified facts. Always double-check important answers. |
| **Knowledge cutoff** | It only knows what it saw in training, unless you add search. |
| **Bias** | It can reflect biases in its training data. |
| **Privacy** | Don't send passwords, personal data or private documents without knowing the data policy. |
| **Cost & limits** | The free tier has **rate limits**. If you see a "quota" or "429" error, wait a minute and try again. |

### 🧪 Test it
Ask the bot something it can't know or that is made up, like *"Who won the 2087 World Cup?"* or *"Tell me about the book 'The Silver Lemon Chronicles' by Ravi Menon."* Does it admit it doesn't know, or does it invent an answer?

---
## 🏁 Summary

1. A chatbot = **your code + an API + an LLM** running on Google's servers.
2. The **API key** identifies you. Keep it secret (Colab Secrets).
3. `generate_content()` gives one answer, and `generate_content_stream()` streams it.
4. **LLMs have no memory.** A `chat` session works by resending the history.
5. **System instructions** control personality and rules.
6. **Gradio** gives a chat interface in a few lines.
7. **Grounding** adds live web search.
8. LLMs can **hallucinate**, so verify what matters.

### 📚 Glossary

| Term | Simple meaning |
|---|---|
| **API** | A way for programs to talk to a service |
| **API key** | Your password for using the API |
| **LLM** | Large Language Model, a huge Transformer that predicts the next token |
| **Prompt** | The input/instruction you give the model |
| **System instruction** | Hidden rules that shape the bot's behaviour |
| **Streaming** | Receiving the answer in pieces as it is written |
| **Context window** | The maximum text the model can consider at once |
| **Hallucination** | A confident but false answer |
| **Grounding** | Backing answers with real data, like web search |

### 🚀 Take-home challenges
1. Build a **specialist bot** (study buddy, recipe helper, interview coach) with a good system instruction.
2. Make the bot answer **in Malayalam, Tamil or Hindi** (just add it to the system instruction!).
3. Upload an image or PDF and ask questions about it. Gemini is **multimodal**, so look up `types.Part.from_bytes`.
4. Try a newer model by changing `MODEL_NAME` and compare the quality of the answers.

---
*Day 1: ML basics → Day 2: Neural networks → Day 3: Transformers → Day 4: your own AI chatbot.* 🎓